# Notebook 04 — Práctica

Cuarto y último sub-bloque del Tema 06. **6 ejercicios graduales** sobre Northwind que combinan bloques anónimos, control de flujo y procedimientos almacenados.

Cada ejercicio sigue el patrón ya conocido:

1. **Enunciado** en markdown.
2. **Celda vacía** para tu solución.
3. **Solución** colapsable (`<details>`) — ábrela solo cuando hayas intentado.

Dos niveles:

- **Fácil (1-4):** bloques anónimos con `IF`/`CASE`, variables, `FOR`/`WHILE`; la salida se muestra vía tabla temporal + `SELECT`.
- **Procedimientos (5-6):** `CREATE PROCEDURE` + `CALL`, validaciones, decisión procedimiento vs SQL puro.

## Setup

In [ ]:
# Setup — instala JupySQL si hace falta (Colab trae ipython-sql, no JupySQL).
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel (Entorno de ejecución → Reiniciar sesión)")
    print("  y vuelve a correr esta celda y las siguientes.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine
# Devuelve cada query como DataFrame de pandas (mejor render en Colab, y
# el resultado es directamente manipulable con pandas).
%config SqlMagic.autopandas = True

## Nivel fácil (1-4)

### Ejercicio 1 — Saludo según hora del día

Crea un bloque anónimo que declare una variable `hora` con el valor de `EXTRACT(hour FROM CURRENT_TIMESTAMP)` y, según la hora, registre en una **tabla temporal** (mostrada con `SELECT`):

- `'Buenos días'` si la hora es menor a 12
- `'Buenas tardes'` si está entre 12 y 18 (inclusive)
- `'Buenas noches'` si es mayor a 18

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
DROP TABLE IF EXISTS salida;
CREATE TEMP TABLE salida (mensaje TEXT);

DO $$
DECLARE
    hora INTEGER := EXTRACT(hour FROM CURRENT_TIMESTAMP);
    msg  TEXT;
BEGIN
    IF hora < 12 THEN
        msg := 'Buenos días';
    ELSIF hora <= 18 THEN
        msg := 'Buenas tardes';
    ELSE
        msg := 'Buenas noches';
    END IF;
    INSERT INTO salida VALUES (msg || ' (hora=' || hora || ')');
END
$$;

SELECT * FROM salida;
```
</details>

### Ejercicio 2 — Sumatoria del 1 al N

Bloque anónimo que calcule la suma del 1 al 100 usando un `FOR` loop y la muestre con un `SELECT` desde una tabla temporal. Verifica que el resultado sea 5050.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
DROP TABLE IF EXISTS salida;
CREATE TEMP TABLE salida (resultado TEXT);

DO $$
DECLARE
    suma INTEGER := 0;
BEGIN
    FOR i IN 1..100 LOOP
        suma := suma + i;
    END LOOP;
    INSERT INTO salida VALUES ('Suma 1 a 100 = ' || suma);
END
$$;

SELECT * FROM salida;
```

Nota: esto se podría hacer en SQL puro con `SELECT SUM(i) FROM generate_series(1, 100) AS i;` — el ejercicio es para practicar el `FOR` loop.
</details>

### Ejercicio 3 — Categorizar conteo de clientes

Bloque anónimo que cuente clientes en `dim_customer` y, según el conteo, registre en una tabla temporal:

- `'Base pequeña (<50)'`
- `'Base mediana (50-200)'`
- `'Base grande (>200)'`

Usa `SELECT ... INTO` para asignar el conteo a la variable.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
DROP TABLE IF EXISTS salida;
CREATE TEMP TABLE salida (mensaje TEXT);

DO $$
DECLARE
    total INTEGER;
    msg   TEXT;
BEGIN
    SELECT COUNT(*) INTO total FROM northwind_dwh.dim_customer;

    IF total < 50 THEN
        msg := 'Base pequeña (<50)';
    ELSIF total <= 200 THEN
        msg := 'Base mediana (50-200)';
    ELSE
        msg := 'Base grande (>200)';
    END IF;
    INSERT INTO salida VALUES (msg || ': ' || total || ' clientes');
END
$$;

SELECT * FROM salida;
```
</details>

### Ejercicio 4 — Listar productos por categoría

Bloque anónimo que itere sobre las categorías de `dim_product` y registre **una fila por categoría** (nombre + número de productos) en una tabla temporal, mostrada con `SELECT`. Usa `FOR ... IN SELECT`.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
DROP TABLE IF EXISTS salida;
CREATE TEMP TABLE salida (categoria TEXT, productos INTEGER);

DO $$
DECLARE
    fila RECORD;
BEGIN
    FOR fila IN
        SELECT category_name, COUNT(*) AS productos
          FROM northwind_dwh.dim_product
         GROUP BY category_name
         ORDER BY productos DESC
    LOOP
        INSERT INTO salida VALUES (fila.category_name, fila.productos);
    END LOOP;
END
$$;

SELECT * FROM salida;
```
</details>

## Nivel — procedimientos (5-6)

### Ejercicio 5 — Procedimiento `reportar_top_clientes`

Crea un **procedimiento** que reciba un parámetro `p_n INTEGER` y un parámetro `OUT p_total NUMERIC`. El procedimiento debe:

1. Registrar en una tabla los top N clientes por ventas netas (nombre + total).
2. Dejar el gran total en `p_total`.

Invócalo con `CALL reportar_top_clientes(5, NULL);` y muestra la tabla con `SELECT`.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
DROP TABLE IF EXISTS top_clientes;
CREATE TEMP TABLE top_clientes (cliente TEXT, total NUMERIC);

CREATE OR REPLACE PROCEDURE reportar_top_clientes(
    p_n     INTEGER,
    OUT p_total NUMERIC
) AS $$
DECLARE
    fila RECORD;
BEGIN
    p_total := 0;

    FOR fila IN
        SELECT dc.company_name, SUM(fs.line_total) AS total
          FROM northwind_dwh.fact_sales fs
          JOIN northwind_dwh.dim_customer dc USING (customer_key)
         GROUP BY dc.company_name
         ORDER BY 2 DESC
         LIMIT p_n
    LOOP
        INSERT INTO top_clientes VALUES (fila.company_name, ROUND(fila.total, 2));
        p_total := p_total + fila.total;
    END LOOP;
END;
$$ LANGUAGE plpgsql;

CALL reportar_top_clientes(5, NULL);
SELECT * FROM top_clientes;
```

El `OUT p_total` queda con el gran total (lo devuelve el `CALL`); la tabla `top_clientes` muestra el detalle por cliente.
</details>

### Ejercicio 6 — Procedimiento de validación de DWH

Crea un procedimiento `validar_dwh()` que verifique tres invariantes y **registre el resultado de cada una** (`OK`/`FALLA` + detalle) en una tabla temporal:

1. Conteo de `fact_sales` coincide con conteo de `northwind_oltp.order_details`.
2. Ninguna fila de `fact_sales` tiene `customer_key`, `product_key` o `employee_key` NULL.
3. `SUM(line_total)` en DWH es ≈ `SUM(quantity * unit_price * (1 - discount))` en OLTP (tolerancia ±$10 por la corrección REAL→NUMERIC).

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
DROP TABLE IF EXISTS reporte_validacion;
CREATE TEMP TABLE reporte_validacion (invariante TEXT, estado TEXT, detalle TEXT);

CREATE OR REPLACE PROCEDURE validar_dwh() AS $$
DECLARE
    c_fact INTEGER; c_oltp INTEGER; c_nulls INTEGER; s_dwh NUMERIC; s_oltp NUMERIC;
BEGIN
    -- 1. Conteo
    SELECT COUNT(*) INTO c_fact FROM northwind_dwh.fact_sales;
    SELECT COUNT(*) INTO c_oltp FROM northwind_oltp.order_details;
    INSERT INTO reporte_validacion VALUES (
        'Conteo fact = order_details',
        CASE WHEN c_fact = c_oltp THEN 'OK' ELSE 'FALLA' END,
        FORMAT('fact=%s, oltp=%s', c_fact, c_oltp));

    -- 2. FKs obligatorias
    SELECT COUNT(*) INTO c_nulls
      FROM northwind_dwh.fact_sales
     WHERE customer_key IS NULL OR product_key IS NULL OR employee_key IS NULL;
    INSERT INTO reporte_validacion VALUES (
        'Sin FK NULL en fact_sales',
        CASE WHEN c_nulls = 0 THEN 'OK' ELSE 'FALLA' END,
        FORMAT('%s filas con FK NULL', c_nulls));

    -- 3. SUM consistente
    SELECT SUM(line_total) INTO s_dwh FROM northwind_dwh.fact_sales;
    SELECT SUM(quantity * unit_price * (1 - discount)) INTO s_oltp
      FROM northwind_oltp.order_details;
    INSERT INTO reporte_validacion VALUES (
        'SUM DWH = SUM OLTP (+-10)',
        CASE WHEN ABS(s_dwh - s_oltp) <= 10 THEN 'OK' ELSE 'FALLA' END,
        FORMAT('DWH=%s, OLTP=%s, delta=%s', ROUND(s_dwh,2), ROUND(s_oltp,2), ROUND(s_dwh-s_oltp,2)));
END;
$$ LANGUAGE plpgsql;

CALL validar_dwh();
SELECT * FROM reporte_validacion;
```
</details>

## Limpieza

Si quieres eliminar las funciones y procedimientos que creaste durante los ejercicios:

In [ ]:
%%sql
DROP PROCEDURE IF EXISTS reportar_top_clientes(INTEGER, NUMERIC);
DROP PROCEDURE IF EXISTS validar_dwh();
DROP TABLE     IF EXISTS salida, top_clientes, reporte_validacion;

## Cierre del Tema 06

Lo que construiste a lo largo de los cuatro notebooks:

| Notebook | Tema |
|---|---|
| **01** | Bloques anónimos `DO $$`, variables, `RAISE`, `SELECT INTO`, `PERFORM`, manejo de excepciones |
| **02** | Control de flujo: `IF`, `CASE`, `FOR` (rango/query), `WHILE` |
| **03** | Procedimientos: `CREATE PROCEDURE`, `CALL`, `DROP`, parámetros `IN`/`OUT`/`INOUT`, cursores |
| **04** | Práctica integrada: 6 ejercicios graduales |

**Lo que sigue en el Tema 07:** funciones de ventana — `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `LEAD`, sumas acumulativas, promedios móviles. Aritmética por filas que ni `GROUP BY` ni PL/pgSQL hacen tan bien.

---

<p align="center">
<a href="03_procedimientos_y_cursores.ipynb">← Anterior: Notebook 03</a> | <a href="Readme.md">Volver al índice</a> | <a href="../Tema-07/Readme.md">Siguiente: Tema 07 →</a>
</p>